In [1]:
#!pip install torch torchvision torchaudio
#!pip install qutip
#!pip install sympy

In [ ]:
from src.gate_pinn_module import *
from sympy.physics.quantum import TensorProduct
from sympy.physics.matrices import msigma
import qutip as qt
from itertools import product

In [3]:
#|0>, |1>
ket_0, ket_1 = basis(2)

ket_00 = TensorProduct(ket_0, ket_0)
ket_01 = TensorProduct(ket_0, ket_1)
ket_10 = TensorProduct(ket_1, ket_0)
ket_11 = TensorProduct(ket_1, ket_1)

sig_low = ket_1*sympy_dag(ket_0)
sig_up  = ket_0*sympy_dag(ket_1)
sig_x = sig_up + sig_low
sig_y = -sympy.I*(sig_up-sig_low)
sig_z = ket_0*sympy_dag(ket_0) - ket_1*sympy_dag(ket_1)

sig_z1 = TensorProduct(sig_z, sympy.eye(2))
sig_z2 = TensorProduct(sympy.eye(2), sig_z)
sig_x1 = TensorProduct(sig_x, sympy.eye(2))
sig_x2 = TensorProduct(sympy.eye(2), sig_x)
sig_y1 = sympy.physics.quantum.TensorProduct(sig_y, sympy.eye(2))
sig_y2 = sympy.physics.quantum.TensorProduct(sympy.eye(2), sig_y)
sig_low_1 = TensorProduct(sig_low, sympy.eye(2))
sig_low_2 = TensorProduct(sympy.eye(2), sig_low)
sig_up_1 = TensorProduct(sig_up, sympy.eye(2))
sig_up_2 = TensorProduct(sympy.eye(2), sig_up)

In [4]:
#Funciones de control a resolver mediante PINN
control, init_control = control_create(symbols=[ "ex1", "ey1", "ez1", "ex2", "ey2", "ez2"], init_control=[0,0,0,0,0,0])

#Hamiltoniano dependiente del tiempo
HL = control["ex1"]*sig_x1 + control["ey1"]*sig_y1 + control["ez1"]*sig_z1 + control["ex2"]*sig_x2 + control["ey2"]*sig_y2 + control["ez2"]*sig_z2

In [5]:
#Compuerta CNOT
CNOT = TensorProduct(ket_0*ket_0.T, sympy.eye(2)) + TensorProduct(ket_1*ket_1.T, sig_x)

In [6]:
#Hamiltoniano total (acople XX)
H = TensorProduct(sig_x, sig_x) + HL

In [7]:
#Los estados iniciales corresponderán con la base {|00>, |01>, |10>, |11>}
rho_00_00 = ket_00*sympy_dag(ket_00)
rho_01_01 = ket_01*sympy_dag(ket_01)
rho_10_10 = ket_10*sympy_dag(ket_10)
rho_11_11 = ket_11*sympy_dag(ket_11)

In [8]:
#Estados iniciales arbitrarios
psi_arb= (1/np.sqrt(2))*(ket_00+ket_10)
rho_arb= psi_arb*sympy_dag(psi_arb)
rho_cnot_arb = CNOT*rho_arb*sympy_dag(CNOT)

#Estados para entrenamiento por transferencia
psi_arb_2 = (1/np.sqrt(2))*(ket_01-ket_10)
rho_arb_2 = psi_arb_2*sympy_dag(psi_arb_2)
rho_cnot_arb_2 = CNOT*rho_arb_2*sympy_dag(CNOT)

psi_arb_3 = (1/np.sqrt(2))*(ket_00+sympy.I*ket_11)
rho_arb_3 = psi_arb_3*sympy_dag(psi_arb_3)
rho_cnot_arb_3 = CNOT*rho_arb_3*sympy_dag(CNOT)


In [9]:
#Condiciones iniciales. Debe coincidir en orden y cantidad con los estados iniciales.
rho_init = np.array([
    rho_00_00,
    rho_01_01,
    rho_10_10,
    rho_11_11,
    rho_arb,
    rho_arb_2,
    rho_arb_3
])
#Condiciones objetivo impuesto al sistema. Debe coincidir en orden y cantidad con los estados iniciales.
rho_target = np.array([
    rho_00_00,
    rho_01_01,
    rho_11_11,
    rho_10_10,
    rho_cnot_arb,
    rho_cnot_arb_2,
    rho_cnot_arb_3
])

In [10]:
#Estados aleatorios para evaluar la fidelidad promedio
estados = 500
qubit_1 = [qt.rand_ket(2) for i in range(estados)] # Genero kets aleatorios para qubit 1
qubit_2 = [qt.rand_ket(2) for i in range(estados)] # Genero kets aleatorios para qubit 2
qubits_12 = [qt.tensor(kets_q1, kets_q2) for kets_q1, kets_q2 in zip(qubit_1, qubit_2)] # Producto tensorial de los qubits

# Aplico la CNOT y calculo los estados iniciales en representación de matrices
U = qt.tensor(qt.ket2dm(qt.basis(2,0)), qt.qeye(2)) + qt.tensor(qt.ket2dm(qt.basis(2,1)), qt.sigmax())

U_rho=[]
rho0=[]
for ket in qubits_12:
    rho=ket*ket.dag()
    rho0.append(rho)
    U_rho.append(U*rho*U.dag())

In [11]:
#valores de barrido 
lr=np.linspace(1e-4,1e-3,10)  #Tasa de aprendizaje
ep=np.arange(5000,30001,5000)  #Epocas de entrenamiento
n=np.arange(100,501,50)  #neuronas
hl=np.arange(2,10,2)  #capas ocultas
#eta=np.linspace(1e-4,1,200)  #peso del estado objetivo
chi=np.linspace(1e-5,1e-3,5)  #peso de la regularizacion

In [ ]:

Sweep = []
#for lr_i, ep_i, n_i, hl_i, chi_i in product(lr, ep, n, hl, chi):
for lr_i, n_i in product(lr,n):
    
    ################################################################################
    #Entrenamiento para la PINN
    ################################################################################   

    pinn_properties = {
        "learning_rate": lr_i,
        "epochs": 2e4,
        "batch_size": 500,
        "neurons": n_i,
        "hidden_layers": 4,
        "time_config": [0, 5, 500], 
        "eta": 0.1, # Peso de la condición del estado objetivo,
        "eta_sc": 1, # Peso de la condición de la traza
        "chi": 1e-3, # Hiperparámetro de la regularización l2
        "debug_model": False,
        "num_workers": 0, #Configurar 4 para GPU o 0 para CPU, mejora el rendimiento.
        "loss_gate": True
    }
    #Clase solución
    solution = Solver(H, control, init_control, rho_init, rho_target, pinn_properties)

    #Entrenamiento de la red
    solution.train_neural_network()

    #Vector de tiempo para evaluar el control aprendido
    t_test = torch.linspace(0.0, 5, 500).reshape(-1, 1)
    #t_test.requires_grad=True
    t_net = t_test.detach().numpy()

    ex1 = solution.eval_component(t_test, "ex1")
    ey1 = solution.eval_component(t_test, "ey1")
    ez1 = solution.eval_component(t_test, "ez1")
    ex2 = solution.eval_component(t_test, "ex2")
    ey2 = solution.eval_component(t_test, "ey2")
    ez2 = solution.eval_component(t_test, "ez2")
    
    ###########################################################################
    #Fidelidad promedio usando qutip 
    ###########################################################################

    #Controles en qutip
    Q_ex1=ex1.transpose().squeeze()
    Q_ey1=ey1.transpose().squeeze()
    Q_ez1=ez1.transpose().squeeze()
    Q_ex2=ex2.transpose().squeeze()
    Q_ey2=ey2.transpose().squeeze()
    Q_ez2=ez2.transpose().squeeze()
    
    #Hamiltoniano
    H0=qt.tensor(qt.sigmax(), qt.sigmax())
    Q_H=[H0,[qt.tensor(qt.sigmax(), qt.qeye(2)), Q_ex1],
            [qt.tensor(qt.sigmay(), qt.qeye(2)), Q_ey1],
            [qt.tensor(qt.sigmaz(), qt.qeye(2)), Q_ez1],
            [qt.tensor(qt.qeye(2), qt.sigmax()), Q_ex2],
            [qt.tensor(qt.qeye(2), qt.sigmay()), Q_ey2],
            [qt.tensor(qt.qeye(2), qt.sigmaz()), Q_ez2]]
    
    #Tiempo
    times=np.linspace(0.0,5.0,500)
    
    #Solución de la dinamica cerrada controlada para cada estado aleatorio
    solution_H=[]
    for rho in rho0:
        result= qt.mesolve(Q_H, rho, times, [], [])
        solution_H.append(result.states[-1])
    
    #Fidelidad promedio en T=5 
    fid_H=[]
    for i in range(estados):
        f_H=qt.fidelity(solution_H[i], U_rho[i])
        fid_H.append(f_H**2)
    
    fid_prom_H=sum(fid_H)/estados
    
    #Agregamos los parametros, control y fidelidad a un diccionario
    Sweep.append({
        "learning rate": lr_i,
        "epochs": 2e4,
        "neurons": n_i,
        "hidden_layers": 4, #hl_i,
        "chi": 1e-3,#chi_i,
        "A. fidelity": fid_prom_H,
        "ex1": ex1,
        "ey1": ey1,
        "ez1": ez1,
        "ex2": ex2,
        "ey2": ey2,
        "ez2": ez2,
        "Loss": solution.loss_history,
        "L_model": solution.loss_model,
        "L_control": solution.loss_control,
        "L_const": solution.loss_const,
        "L_reg" : solution.loss_reg  
    })

#Almacenamos el barrido en un archivo binario
filename = f"CNOT_lr{lr[0]:.0e}-{lr[-1]:.0e}_neurons{n[0]}-{n[-1]}_Hxx.npy"
np.save(filename, Sweep, allow_pickle=True)